Vanguard Project - Vanguard Digital Experience Analysis

Dataset - https://raw.githubusercontent.com/data-bootcamp-v4/lessons/refs/heads/main/5_6_eda_inf_stats_tableau/project/files_for_project/df_final_web_data_pt_1.txt

In [1]:
import pandas as pd

url_pt1 = "https://raw.githubusercontent.com/data-bootcamp-v4/lessons/refs/heads/main/5_6_eda_inf_stats_tableau/project/files_for_project/df_final_web_data_pt_1.txt"

df1 = pd.read_csv(url_pt1, sep=",")

df1.head(3)


,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22


Inspect the variables available in each dataset.

In [2]:
# Size of dataset

print(df1.shape)

(343141, 5)


In [3]:
# Check column names and data types
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 343141 entries, 0 to 343140
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   client_id     343141 non-null  int64
 1   visitor_id    343141 non-null  str  
 2   visit_id      343141 non-null  str  
 3   process_step  343141 non-null  str  
 4   date_time     343141 non-null  str  
dtypes: int64(1), str(4)
memory usage: 13.1 MB


In [4]:
# Check for missing values
df1.isnull().sum()

client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
dtype: int64

In [5]:
# Check for duplicate rows
print(df1.duplicated().sum()) 

2095


In [6]:
# Actual duplicate rows 
df1[df1.duplicated()] 

,client_id,visitor_id,visit_id,process_step,date_time
365,9358626,148475529_63668223960,223297395_36250329195_832161,start,2017-04-28 12:27:28
368,5376894,622509328_44408201172,688984457_43441834354_912755,start,2017-04-28 14:20:01
383,438567,923318467_50708963627,330543236_29863358529_771432,start,2017-04-28 02:52:29
785,1423857,716830992_81930238382,568819166_80596727019_75889,confirm,2017-04-10 22:44:58
1816,372523,730241475_25771921122,25331354_74479847014_854987,start,2017-04-27 10:51:26
...,...,...,...,...,...
342621,1853345,831456815_77348510380,596026060_87178689141_910423,step_3,2017-04-13 19:39:24
342677,1251269,791822615_78146852845,763845944_55757914723_70888,start,2017-04-28 10:20:33
342679,1251269,791822615_78146852845,763845944_55757914723_70888,start,2017-04-28 10:18:20
342685,8350388,304440669_31040182888,143359398_96671536668_299057,start,2017-04-28 12:44:21


In [7]:
# Check unique values

df1['process_step'].unique() 


<StringArray>
['step_3', 'step_2', 'step_1', 'start', 'confirm']
Length: 5, dtype: str

In [8]:
# Rows per stage ?

df1['process_step'].value_counts()  

process_step
start      108910
step_1      73432
step_2      61768
step_3      53628
confirm     45403
Name: count, dtype: int64

In [18]:
# just checking column date

df1['date_time'].dtype

<StringDtype(storage='python', na_value=nan)>

In [17]:
print("Missing:", df1['date_time'].isna().sum())

print("Unique:", df1['date_time'].nunique())

print("Duplicates:", df1['date_time'].duplicated().sum())

Missing: 0
Unique: 282666
Duplicates: 60475


Which identifier should be used for each type of analysis?

In [19]:
# Check for unique values for column client, visitor and visit

print("Unique client_id:", df1["client_id"].nunique())
print("Unique visitor_id:", df1["visitor_id"].nunique())
print("Unique visit_id:", df1["visit_id"].nunique())
print("Rows:", len(df1))

Unique client_id: 58391
Unique visitor_id: 62936
Unique visit_id: 75256
Rows: 343141


One client has many visit --> Unique client_id: 58391 != Unique visit_id: 75256

In [20]:
# Check if one client has multiple visits

df1.groupby("client_id")["visit_id"].nunique().sort_values(ascending=False).head()

client_id
2128341    21
9852814    15
9008485    13
9663519    13
9018602    12
Name: visit_id, dtype: int64

In [ ]:
Customer journey --> A client can return multiple times and create multiple visits.

In [29]:
# Client --> Visitor

df1.groupby("client_id")["visitor_id"].nunique().value_counts().sort_index()

visitor_id
1     54079
2      3742
3       423
4        92
5        22
6        17
7        13
8         2
12        1
Name: count, dtype: int64

Verifying Hierarchy 

In [30]:
# Visit --> Client 

df1.groupby("visit_id")["client_id"].nunique().value_counts()

client_id
1    74866
2      390
Name: count, dtype: int64

A visit should ideally belong to only one client.

In [31]:
# Visit --> Visitor 

df1.groupby("visit_id")["visitor_id"].nunique().value_counts()


visitor_id
1    75256
Name: count, dtype: int64

One visit id per visitor

In [21]:
# one visitor belongs to multiple clients

df1.groupby("visitor_id")["client_id"].nunique().value_counts()

client_id
1    62346
2      586
3        3
4        1
Name: count, dtype: int64

Visitor_id are associated with multiple clients therefore should be avoided as a primary customer identifier (could be due to shared devices, authentication, household accounts) 

In [23]:
# Checking for exact duplicates

df1[df1.duplicated(keep=False)].sort_values(
["client_id", "visit_id"]
)

,client_id,visitor_id,visit_id,process_step,date_time
340965,5459,128322125_76706471522,86991584_85213257843_356594,start,2017-04-27 11:53:56
340966,5459,128322125_76706471522,86991584_85213257843_356594,start,2017-04-27 11:53:56
192508,13009,360435735_44466390795,377986493_6391607481_598681,step_3,2017-03-29 12:10:34
192509,13009,360435735_44466390795,377986493_6391607481_598681,step_3,2017-03-29 12:10:34
59443,36109,306603996_85757417975,323091457_98221684598_337461,confirm,2017-03-29 21:55:55
...,...,...,...,...,...
245847,9995203,550384864_76045959210,379204533_66746802931_987441,start,2017-04-29 15:59:34
256561,9997801,500453195_9636586711,829652786_55390211114_665695,start,2017-04-28 10:59:18
256562,9997801,500453195_9636586711,829652786_55390211114_665695,start,2017-04-28 10:59:18
119532,9998346,292425655_16607136645,189177304_69869411700_783154,step_3,2017-03-29 15:37:28


In [24]:
# Check repeated process steps within a visit

duplicates_steps = df1[
    df1.duplicated(
        subset=["client_id", "visit_id", "process_step"],
        keep=False
)
]

duplicates_steps.sort_values(
["client_id", "visit_id", "process_step", "date_time"]
)

,client_id,visitor_id,visit_id,process_step,date_time
245500,722,919259913_64837298108,984487154_55831795985_521110,start,2017-04-19 14:50:10
245496,722,919259913_64837298108,984487154_55831795985_521110,start,2017-04-19 14:56:16
245499,722,919259913_64837298108,984487154_55831795985_521110,step_1,2017-04-19 14:50:20
245497,722,919259913_64837298108,984487154_55831795985_521110,step_1,2017-04-19 14:51:12
245495,722,919259913_64837298108,984487154_55831795985_521110,step_1,2017-04-19 14:56:18
...,...,...,...,...,...
260235,9999333,97912417_51227849676,975376198_67887789298_440036,start,2017-03-29 09:52:23
107166,9999729,843385170_36953471821,493310979_9209676464_421146,start,2017-04-20 14:21:27
107163,9999729,843385170_36953471821,493310979_9209676464_421146,start,2017-04-20 14:28:57
274436,9999839,948322592_28990736246,715530422_68620416793_515645,start,2017-03-29 12:08:55


In [26]:
# How often it happens
(
df1.groupby(["client_id", "visit_id", "process_step"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

client_id  visit_id                      process_step
2313292    712824876_8175482950_365042   step_2          24
5427647    339824895_73123393347_159684  step_2          23
2206737    292630414_4329877712_308107   start           23
2376317    552148174_59201792774_449534  step_1          21
3182757    447501825_86900283385_427480  start           19
7699482    598603763_85483459636_600247  start           19
8768579    181601848_48411429197_546198  start           19
8678658    704585335_84106003655_168533  start           19
4900077    765667444_71098075557_436330  step_1          18
7795550    428529357_6959155752_124163   start           17
8406607    40675776_79064823233_179200   start           17
5595824    782512439_52715344954_564286  start           17
2313292    712824876_8175482950_365042   step_3          16
5933262    634213905_6270988650_650282   step_3          16
843228     542291007_99212333873_758727  start           16
8360801    530163740_52602749918_87343   start

In [27]:
# inspect time differences 

df1["date_time"] = pd.to_datetime(df1["date_time"])

In [28]:
sample = duplicates_steps.sort_values(
    ["client_id", "visit_id", "process_step", "date_time"]
)

sample.head(10)

,client_id,visitor_id,visit_id,process_step,date_time
245500,722,919259913_64837298108,984487154_55831795985_521110,start,2017-04-19 14:50:10
245496,722,919259913_64837298108,984487154_55831795985_521110,start,2017-04-19 14:56:16
245499,722,919259913_64837298108,984487154_55831795985_521110,step_1,2017-04-19 14:50:20
245497,722,919259913_64837298108,984487154_55831795985_521110,step_1,2017-04-19 14:51:12
245495,722,919259913_64837298108,984487154_55831795985_521110,step_1,2017-04-19 14:56:18
245498,722,919259913_64837298108,984487154_55831795985_521110,step_2,2017-04-19 14:50:56
245494,722,919259913_64837298108,984487154_55831795985_521110,step_2,2017-04-19 14:56:37
62623,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:36:30
62622,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:37:02
62621,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:24


Customer-level analysis

In [33]:
# Unique customers

df1["client_id"].nunique()

# Number of visits per customer
visits_per_client = df1.groupby("client_id")["visit_id"].nunique()

visits_per_client.describe()

count    58391.000000
mean         1.295508
std          0.693226
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: visit_id, dtype: float64

Visit (Session)-level analysis

In [35]:
# Verify visit -> visitor relationship
df1.groupby("visit_id")["visitor_id"].nunique().value_counts()

# Verify visit -> client relationship
df1.groupby("visit_id")["client_id"].nunique().value_counts()

client_id
1    74866
2      390
Name: count, dtype: int64

Event-level analysis

In [36]:
# Repeated process steps
(
df1.groupby(
    ["client_id", "visit_id", "process_step"]
)
    .size()
    .sort_values(ascending=False)
    .head(10)
)

client_id  visit_id                      process_step
2313292    712824876_8175482950_365042   step_2          24
5427647    339824895_73123393347_159684  step_2          23
2206737    292630414_4329877712_308107   start           23
2376317    552148174_59201792774_449534  step_1          21
3182757    447501825_86900283385_427480  start           19
7699482    598603763_85483459636_600247  start           19
8768579    181601848_48411429197_546198  start           19
8678658    704585335_84106003655_168533  start           19
4900077    765667444_71098075557_436330  step_1          18
7795550    428529357_6959155752_124163   start           17
dtype: int64

In [38]:
# check for duplicate

df1.duplicated().sum()

np.int64(2095)

In [10]:
# Basic descriptive stats
df1.describe(include='all')

,client_id,visitor_id,visit_id,process_step,date_time
count,3.431410e+05,343141,343141,343141,343141
unique,NaN,62936,75256,5,282666
top,NaN,699275239_82397698587,712824876_8175482950_365042,start,2017-03-29 11:24:54
freq,NaN,66,61,108910,14
mean,4.996097e+06,NaN,NaN,NaN,NaN
std,2.875839e+06,NaN,NaN,NaN,NaN
min,1.690000e+02,NaN,NaN,NaN,NaN
25%,2.506067e+06,NaN,NaN,NaN,NaN
50%,5.007915e+06,NaN,NaN,NaN,NaN
75%,7.468519e+06,NaN,NaN,NaN,NaN
